Варіант 6
Місто Чернігів

In [1]:
import pandas as pd
import numpy as np

# ==============================================================================
# ПАРАМЕТРИ ВАРІАНТА 6
# Місто: Чернігів
# Базова ціна: 690 грн
# Базова кількість: 4
# ==============================================================================
VARIANT = 6
CITY = "Чернігів"
BASE_PRICE = 690
BASE_QTY = 4

# Зафіксуємо seed для відтворюваності результатів
np.random.seed(VARIANT)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 1. Побудова набору клієнтів (10 замовлень)
# ------------------------------------------------------------------------------
cities_list = ["Чернігів", "Київ", "Львів", "Одеса", "Харків"]

data = {
    "клієнт_id": range(1, 11),
    "вік": [22, 45, 31, 58, 29, 64, 38, 19, 50, 41],
    "дохід": [15000, 48000, 28000, 65000, 22000, 52000, 35000, 12000, 41000, 33000],
    "кількість": [2, 5, 4, 3, 6, 4, 2, 1, 5, 4],
    "місто_доставки": [
        "Чернігів", "Київ", "Львів", "Одеса", "Чернігів",
        "Харків", "Київ", "Львів", "Чернігів", "Одеса"
    ]
}

df = pd.DataFrame(data)

print("=== Завдання 1: Початковий DataFrame замовлень ===")
print(df)
print("\n")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 2. Похідні змінні
# ------------------------------------------------------------------------------
df["сума_замовлення"] = BASE_PRICE * df["кількість"]
df["частка_доходу"] = df["сума_замовлення"] / df["дохід"]

print("=== Завдання 2: DataFrame з похідними змінними ===")
print(df[["клієнт_id", "дохід", "кількість", "сума_замовлення", "частка_доходу"]])
print("\n")

explanation_task2 = """
[Аналіз до Завдання 2]
Чому 'частка_доходу' корисніша для аналізу поведінки клієнтів:
Показник 'частка_доходу' нормує витрати клієнта відносно його фінансових можливостей.
Наприклад, замовлення на суму 2760 грн для клієнта з доходом 12 000 грн становить 23% місячного бюджету (значне, обдумане придбання),
тоді як для клієнта з доходом 65 000 грн те саме замовлення становить лише 4.2% (дрібна, рутинна покупка).
Аналіз виключно 'суми_замовлення' або 'доходу' окремо не дає змоги оцінити рівень фінансового навантаження покупця та його чутливість до цін.
"""
print(explanation_task2)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 3. Дискретизація (cut vs qcut)
# ------------------------------------------------------------------------------
df["вік_cut"] = pd.cut(df["вік"], bins=3)
df["вік_qcut"] = pd.qcut(df["вік"], q=3)

print("=== Завдання 3: Розподіл за pd.cut() (інтервали однакової довжини) ===")
print(df["вік_cut"].value_counts().sort_index())

print("\n=== Завдання 3: Розподіл за pd.qcut() (однакова кількість спостережень) ===")
print(df["вік_qcut"].value_counts().sort_index())
print("\n")

explanation_task3 = """
[Аналіз до Завдання 3]
Чому кількість клієнтів у групах відрізняється між cut та qcut:
1. pd.cut() діє за принципом рівної ДОВЖИНИ інтервалів: діапазон від min(19) до max(64) [різниця 45 років] ділиться на 3 рівні відрізки по 15 років (18.955-34, 34-49, 49-64). Кількість спостережень у цих діапазонах залежить від того, як саме розподілені дані (у нас потрапило 4, 3 та 3 клієнти).
2. pd.qcut() діє за принципом рівної КІЛЬКОСТІ (квантилів): він розбиває масив так, щоб у кожній з 3 груп було приблизно по 3.33 клієнти (у результаті маємо групи по 4, 3 і 3 клієнти з нерівними за довжиною віковими інтервалами).
Якщо б дані мали сильний зсув (наприклад, більшість клієнтів були б молодими), cut дав би порожні або переповнені бакети, тоді як qcut завжди збалансовує розмір груп.
"""
print(explanation_task3)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 4. Масштабування (Min-Max normalization & Z-score standardization)
# ------------------------------------------------------------------------------
# Min-Max Scaling: (x - min) / (max - min)
df["вік_minmax"] = (df["вік"] - df["вік"].min()) / (df["вік"].max() - df["вік"].min())
df["дохід_minmax"] = (df["дохід"] - df["дохід"].min()) / (df["дохід"].max() - df["дохід"].min())

# Standard (Z-score) Scaling: (x - mean) / std
df["вік_zscore"] = (df["вік"] - df["вік"].mean()) / df["вік"].std()
df["дохід_zscore"] = (df["дохід"] - df["дохід"].mean()) / df["дохід"].std()

print("=== Завдання 4: Порівняння результатів масштабування ===")
scaled_cols = [
    "вік", "дохід",
    "вік_minmax", "дохід_minmax",
    "вік_zscore", "дохід_zscore"
]
print(df[scaled_cols].head())
print("\n")

explanation_task4 = """
[Аналіз до Завдання 4]
Чому невимасштабовані вік і дохід не можна напряму порівнювати у масштабочутливих алгоритмах:
Вік вимірюється в десятках (18–64 років), а дохід — у десятках тисяч (12 000–65 000 грн).
У метричних алгоритмах (наприклад, k-NN або K-Means, що обчислюють евклідову відстань), абсолютні різниці у доходах (тисячі гривень) повністю домінуватимуть над різницями у віці (одиниці років). Алгоритм вважатиме вік неістотним шумом.
Приведення обидвох змінних до єдиної шкали [0, 1] або до середнього 0 та std 1 вирівнює їхній фактичний ваговий внесок у розрахунки.
"""
print(explanation_task4)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 5. Індикаторне (One-Hot) кодування
# ------------------------------------------------------------------------------
city_dummies = pd.get_dummies(df["місто_доставки"], prefix="місто", dtype=int)
df_final = pd.concat([df, city_dummies], axis=1)

print("=== Завдання 5: Таблиця з One-Hot кодуванням міста доставки ===")
dummy_cols = [col for col in df_final.columns if col.startswith("місто_")]
print(df_final[["клієнт_id", "місто_доставки"] + dummy_cols])
print("\n")

explanation_task5 = """
[Аналіз до Завдання 5]
Чому для міста доставки доречний саме One-Hot, а не порядкове кодування:
Місто доставки є номінальною змінною. Між містами (наприклад, Чернігів, Київ, Львів, Одеса, Харків) немає природного порядкового зв'язку чи ієрархії.
Якщо закодувати їх числами (1, 2, 3, 4, 5), то математичні моделі сприймуть це як арифметичний порядок: наприклад, ніби Харків (5) «більший» або «кращий» за Чернігів (1), чи що середнє між Черніговом (1) і Львовом (3) дорівнює Києву (2).
One-Hot кодування створює окремі незалежні бінарні прапорці (0/1) для кожного міста, виключаючи виникнення помилкових числових залежностей.
"""
print(explanation_task5)

=== Завдання 1: Початковий DataFrame замовлень ===
   клієнт_id  вік  дохід  кількість місто_доставки
0          1   22  15000          2       Чернігів
1          2   45  48000          5           Київ
2          3   31  28000          4          Львів
3          4   58  65000          3          Одеса
4          5   29  22000          6       Чернігів
5          6   64  52000          4         Харків
6          7   38  35000          2           Київ
7          8   19  12000          1          Львів
8          9   50  41000          5       Чернігів
9         10   41  33000          4          Одеса


=== Завдання 2: DataFrame з похідними змінними ===
   клієнт_id  дохід  кількість  сума_замовлення  частка_доходу
0          1  15000          2             1380       0.092000
1          2  48000          5             3450       0.071875
2          3  28000          4             2760       0.098571
3          4  65000          3             2070       0.031846
4          5  22000 

# Відповіді на контрольні питання
**Практична робота №9: Перетворення даних (похідні змінні, дискретизація, масштабування, індикаторне кодування)**
**Студент:** Варіант 6 (м. Чернігів)

---

### 1. Поясніть різницю між `pd.cut()` і `pd.qcut()` — коли інтервали однакової довжини, а коли гарантована однакова кількість спостережень?

* **`pd.cut()` (дискретизація за інтервалами / Equal-width binning):**
  Ділить весь діапазон значень від $X_{min}$ до $X_{max}$ на $N$ інтервалів **однакової ширини (довжини)**.
  $$\text{Ширина інтервалу} = \frac{X_{max} - X_{min}}{N}$$
  Кількість спостережень у кожній групі буде різною і залежатиме від фактичного щільності даних у цьому діапазоні.

* **`pd.qcut()` (дискретизація за квантилями / Equal-frequency binning):**
  Розбиває дані так, щоб у кожній із $N$ груп була **однакова кількість спостережень** (з точністю до дублікатів межевих значень).
  При цьому ширина самих інтервалів буде різною: там, де дані розташовані щільно, інтервал буде вузьким, а на розріджених ділянках — широким.

---

### 2. Чому для номінальної змінної (місто) застосовують one-hot кодування, а не порядкове чи просто числове (1, 2, 3)?

Номінальна категоріальна змінна (як-от місто доставки, колір товару чи бренд) **не має природного впорядкування або відношень «більше/менше»**.

Якщо використати порядкове кодування ($1, 2, 3, 4$), алгоритми машинного навчання оброблятимуть ці мітки як дійсні числа і зроблять помилкові припущення:
1. **Імпліцитний порядок:** Модель вирішить, що місто з кодом $4$ перевищує місто з кодом $1$ у 4 рази або є важливішим.
2. **Псевдометрія:** Обчислення відстаней або градієнтів враховуватиме різницю $3 - 1 = 2$, створюючи штучний зв'язок між категоріями, які насправді рівноправні.

**One-hot кодування** перетворює кожне унікальне значення на окремий бінарний вектор ($0$ або $1$), зберігаючи ортогональність та незалежність кожної категорії.

---

### 3. Навіщо взагалі масштабувати змінні перед методом, чутливим до масштабу (дистанційні методи, регресія з регуляризацією)?

Більшість математичних методів працюють із числовими значеннями без урахування їхніх фізичних одиниць вимірювання (роки, гривні, метри).

1. **Дистанційні методи (k-NN, K-Means, PCA, SVM):**
   Розраховують евклідову відстань між точками:
   $$d(p, q) = \sqrt{\sum (p_i - q_i)^2}$$
   Якщо змінна $A$ вимірюється в тисячах (дохід: $50\,000$), а змінна $B$ — у десятках (вік: $30$), то складова $(50\,000_1 - 50\,000_2)^2$ повністю придушить значення $(30_1 - 30_2)^2$. Без масштабування змінна з більшим абсолютним розмахом домінуватиме над іншими.
2. **Регресія з регуляризацією (L1/L2 - Lasso/Ridge):**
   Штрафують модель за величину коефіцієнтів $\beta_i$. Якщо змінні мають різний масштаб, коефіцієнти для змінних із великими значеннями будуть природно малими, а для змінних із малими значеннями — великими. Це призведе до того, що регуляризація штрафуватиме змінні нерівномірно і некоректно.

Масштабування (Min-Max або Z-score) приводить усі ознаки до єдиного діапазону, забезпечуючи їхній рівний внесок.

---

### 4. Чому похідна змінна (наприклад, частка доходу на замовлення) часто несе більше сенсу для аналізу, ніж вихідні складники окремо?

Вихідні змінні фіксують **абсолютні фактори**, тоді як похідні змінні описують **відносний контекст та взаємодію** між ними.

* **Абсолютні значення:** Показник `сума_замовлення = 3000 грн` сам по собі є нейтральним.
* **Відносний контекст:**
  * Для клієнта з `доходом = 15 000 грн` частка становить $20\%$ (високобюджетна, критична покупка).
  * Для клієнта з `доходом = 100 000 грн` частка становить $3\%$ (повсякденні витрати).

Побудова похідної змінної `частка_доходу` явно вводить у модель складову **фінансового навантаження/поведінкової чутливості**, яку лінійні методи не завжди спроможні самостійно витягнути з двох окремих стовпчиків `сума_замовлення` та `дохід`.